In [ ]:
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder \
#     .master("local[*]") \
#     .appName('test') \
#     .getOrCreate()

In [2]:
import pyspark
from pyspark.sql import SparkSession

gcs_connector_path = './gcs-connector-hadoop3-latest.jar'

spark = SparkSession.builder \
        .master("local[*]") \
        .appName('Read From GCS Bucket') \
        .config("spark.jars", gcs_connector_path) \
        .config('spark.jars.packages', 'com.google.cloud.spark:spark-bigquery-with-dependencies_2.12:0.35.0') \
        .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
        .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
        .getOrCreate() 

:: loading settings :: url = jar:file:/home/airflow/.local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/airflow/.ivy2/cache
The jars for the packages stored in: /home/airflow/.ivy2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1bbba446-08b2-4b55-b7ab-16c8e0031df9;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.12;0.35.0 in central
downloading https://repo1.maven.org/maven2/com/google/cloud/spark/spark-bigquery-with-dependencies_2.12/0.35.0/spark-bigquery-with-dependencies_2.12-0.35.0.jar ...
	[SUCCESSFUL ] com.google.cloud.spark#spark-bigquery-with-dependencies_2.12;0.35.0!spark-bigquery-with-dependencies_2.12.jar (2733ms)
:: resolution report :: resolve 48850ms :: artifacts dl 2735ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.12;0.35.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules     

In [3]:
from google.cloud import storage
import os
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

BUCKET_NAME="mnl_accident_pipeline_bucket"
KAGGLE_FOLDER="raw/kaggle"
SCRAPE_FOLDER="raw/scrape"

PHT = ZoneInfo("Asia/Manila")
yesterday = datetime.now(PHT) - timedelta(days=1)

In [16]:
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

In [ ]:
# # list files in gcs
# blobs = list(bucket.list_blobs(prefix=KAGGLE_FOLDER))
# files = [blob for blob in blobs if not blob.name.endswith('/')] 

# files[0].name

# # check if file exists in gcs
# filename = "raw/kaggle/kaggle_historical_data.csv"

# stats = storage.Blob(bucket=bucket, name=filename).exists()
# stats

True

In [17]:
def gcs_check_if_exists(bucket, filepath):
    return storage.Blob(bucket=bucket, name=filepath).exists()

In [70]:
# def gcs_file_read(bucket_name, filepath):
#     df = spark.read \
#         .option("header", True) \
#         .option("inferSchema", True) \
#         .option("multiline", True) \
#         .csv(f"gs://{bucket_name}/{filepath}")
    
#     return df
def gcs_file_read(bucket_name, filepath):
    df = spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .option("multiline", True) \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("ignoreLeadingWhiteSpace", True) \
        .option("ignoreTrailingWhiteSpace", True) \
        .csv(f"gs://{bucket_name}/{filepath}")
    
    return df

In [71]:
# FOR KAGGLE
filename = "raw/kaggle/kaggle_historical_data.csv"

df_kaggle = gcs_file_read(BUCKET_NAME, filename)

# df_kaggle.head(5)

In [73]:
df_kaggle.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Time: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- High_Accuracy: integer (nullable = true)
 |-- Direction: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Lanes_Blocked: double (nullable = true)
 |-- Involved: string (nullable = true)
 |-- Tweet: string (nullable = true)
 |-- Source: string (nullable = true)



In [45]:
from pyspark.sql.functions import col

# df_kaggle.where(col("City").isNull()).show()
df_kaggle.where(col("City").isNull()).count()

187

In [46]:
df_kaggle.where(col("City").isNotNull()).count()

17124

In [74]:
Dict_Null = {col:df_kaggle.filter(df_kaggle[col].isNull()).count() for col in df_kaggle.columns}
Dict_Null

{'Date': 0,
 'Time': 122,
 'City': 187,
 'Location': 23,
 'Latitude': 0,
 'Longitude': 0,
 'High_Accuracy': 0,
 'Direction': 857,
 'Type': 57,
 'Lanes_Blocked': 687,
 'Involved': 432,
 'Tweet': 0,
 'Source': 0}

In [75]:
df_kaggle.select("City").distinct().show()

+-------------+
|         City|
+-------------+
|   ParaÃ±aque|
|      Malabon|
|         null|
|  Mandaluyong|
|     Marikina|
|       Taguig|
|   Pasay City|
|       Manila|
|     San Juan|
|  Makati City|
|  Quezon City|
|Kalookan City|
|    Parañaque|
|   Pasig City|
|      Navotas|
|   Valenzuela|
+-------------+



In [76]:
df_kaggle.select("Direction").distinct().show()

+---------+
|Direction|
+---------+
|      EB.|
|       SB|
|     null|
|    CLARA|
|       EB|
|       WB|
|   CLOSED|
|      DAR|
|      PAX|
|       NB|
+---------+



In [82]:
df_kaggle.describe().show()

+-------+--------+-------------+--------------------+------------------+------------------+-------------------+---------+--------------------+------------------+--------------------+--------------------+--------------------+
|summary|    Time|         City|            Location|          Latitude|         Longitude|      High_Accuracy|Direction|                Type|     Lanes_Blocked|            Involved|               Tweet|              Source|
+-------+--------+-------------+--------------------+------------------+------------------+-------------------+---------+--------------------+------------------+--------------------+--------------------+--------------------+
|  count|   17190|        17125|               17289|             17312|             17312|              17312|    16455|               17255|             16625|               16880|               17312|               17312|
|   mean|    null|         null|                null|  14.5594477546459|120.66679424192225|  0.95563

In [64]:
import pandas as pd

df_pd_kaggle = pd.read_csv("../01_extract/.data/data_mmda_traffic_spatial.csv")

In [65]:
df_pd_kaggle["Direction"].unique()

array(['EB', 'NB', 'SB', 'EB.', 'WB', 'DAR', nan, 'CLOSED', 'PAX',
       'CLARA'], dtype=object)

In [67]:
df_test = pd.read_csv(f"gs://{BUCKET_NAME}/raw/kaggle/kaggle_historical_data.csv")

In [68]:
df_test.head()

,Date,Time,City,Location,Latitude,Longitude,High_Accuracy,Direction,Type,Lanes_Blocked,Involved,Tweet,Source
0,2018-08-20,7:55 AM,Pasig City,ORTIGAS EMERALD,14.586343,121.061481,1,EB,VEHICULAR ACCIDENT,1.0,TAXI AND MC,MMDA ALERT: Vehicular accident at Ortigas Emer...,https://twitter.com/mmda/status/10313302019705...
1,2018-08-20,8:42 AM,Mandaluyong,EDSA GUADIX,14.589432,121.057243,1,NB,STALLED L300 DUE TO MECHANICAL PROBLEM,1.0,L300,MMDA ALERT: Stalled L300 due to mechanical pro...,https://twitter.com/mmda/status/10313462477459...
2,2018-08-20,9:13 AM,Makati City,EDSA ROCKWELL,14.559818,121.040737,1,SB,VEHICULAR ACCIDENT,1.0,SUV AND L300,MMDA ALERT: Vehicular accident at EDSA Rockwel...,https://twitter.com/mmda/status/10313589669896...
3,2018-08-20,8:42 AM,Mandaluyong,EDSA GUADIX,14.589432,121.057243,1,NB,STALLED L300 DUE TO MECHANICAL PROBLEM,1.0,L300,MMDA ALERT: Stalled L300 due to mechanical pro...,https://twitter.com/mmda/status/10313590696535...
4,2018-08-20,10:27 AM,San Juan,ORTIGAS CLUB FILIPINO,14.601846,121.046754,1,EB,VEHICULAR ACCIDENT,1.0,2 CARS,MMDA ALERT: Vehicular accident at Ortigas Club...,https://twitter.com/mmda/status/10313711248424...


In [69]:
df_test["Direction"].unique()

array(['EB', 'NB', 'SB', 'EB.', 'WB', 'DAR', nan, 'CLOSED', 'PAX',
       'CLARA'], dtype=object)